In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()



# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from aac_crud import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Kalgun0903"
host = 'localhost'
port = 27017
db = 'aac'
col = 'animals'

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
#print(len(df.to_dict(orient='records')))
#print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    # Add logo and headers
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height':'10%', 'width':'10%'}),
    html.Center(html.B(html.H1('SNHU CS340: Dashbboard'))),
    html.Center(html.B(html.H2("Welcome Grazioso Salvare's Interactive Dashboard"))),
    html.Center(html.P("Developed by: Renee Cullen - 2026")),
    html.Hr(),
    dcc.RadioItems( # Code for filtering radio buttons
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'WR'},
            {'label': 'Mountain Rescue', 'value': 'MR'},
            {'label': 'Disaster Rescue', 'value': 'DR'},
            {'label': 'Reset', 'value': 'RESET'}
        ],
        value='RESET'
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
    # Set up features for your interactive data table to make it user-friendly for your client
                           editable = False,
                         filter_action = "native",
                         sort_action = "native",
                         sort_mode = "multi",
                         column_selectable = "single",
                         row_selectable = "single",
                         row_deletable = False,
                         selected_columns = [],
                         selected_rows = [0],
                         page_action = "native",
                         page_current = 0,
                         page_size = 10,
                        
                        ),
    html.Br(),
    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
             style={'display' : 'flex'},
             children=[
                 dcc.Graph(
                     id='graph-id',
                     className='col s12 m6',
                 ),
                 html.Div(
                     id='map-id',
                     className='col s12 m6',
                 )
         ])
])


    #############################################
    # Interaction Between Components / Controller
    #############################################

@app.callback(
    [Output('datatable-id','data'),
     Output('datatable-id', 'columns')],
    [Input('filter-type', 'value')]
)

def update_dashboard(filter_type):
    query = {}
    data = []
    columns = []
    
    
## Code to filter interactive data table with MongoDB queries

    if filter_type == 'WR': # Water Rescue
        query = {"animal_type": "Dog", "breed": {"$in":["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
        "sex_upon_outcome": "Intact Female", "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'MR': # Mountain Rescue
        query = {"animal_type": "Dog", "breed":{"$in":["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
        "sex_upon_outcome": "Intact Male", "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'DR': # Disaster Rescue
        query = {"animal_type": "Dog", "breed":{"$in":["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
        "sex_upon_outcome": "Intact Male", "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
                }
    # Fetch data from crud module 
    data = shelter.read(query)
        
    # Remove the MongoDB ObjectId for JSON compatability
    if not data:
        return [[], []]
        
    for doc in data:
        doc.pop('_id', None)
            
     # Create the DataFrame
    df = pd.DataFrame.from_records(data)
            
    # Create columns and return
    columns = [{"name": i, "id": i, "selectable": True} for i in df.columns]
            
    return data, columns

@app.callback(
    Output('graph-id', "figure"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    # Convert viewData to DataFrame
    df = pd.DataFrame.from_records(viewData)
    
    if df.empty:
        return px.pie(title="No Data Selected")
    
    # Group all but the top 10 breeds into 'other'
    top_breeds = df['breed'].value_counts().nlargest(10).index
    df.loc[~df['breed'].isin(top_breeds), 'breed'] = 'other'
    
    # Create pie chart
    fig = px.pie(
        df,
        names='breed',
        title='Preferred Animal Breeds (Top 10)'
    )
    # Format for readability
    fig.update_layout(
        height=600,  # Make chart taller
        margin=dict(t=50, b=50) # Add space for title
    )
    fig.update_traces(
        textposition='inside', # Keep labels inside the slices
        textinfo='percent+label'
    )
    return fig

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    # Handle initialization or empty data
    if not viewData or not index:
        # Default view of Austin, TX if no row is selected
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]
    # Convert table data to Dataframe
    dff= pd.DataFrame(viewData)
    
    # Get the row index (single selection)
    row = index[0]
    
    # Access coordinates using column nmaes to avoid index out-of-bounds errors
    # Use .get() to avoid KeyError if columns are missing
    lat = dff.iloc[row].get('location_lat')
    lon = dff.iloc[row].get('location_long')
    animal_name = dff.iloc[row].get('name')
    breed = dff.iloc[row].get('breed')
    
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat,lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(animal_name),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(animal_name),
                    html.H2("Breed"),
                    html.P(breed)
                ])
            ])
        ])
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='external', debug=True)

Dash app running on https://marcosultan-performtitanic-3000.codio.io/proxy/8050/
